# 01 — Aquisição de Dados & Preprocessamento

**Runtime: CPU** (Colab gratuito ou local). Roda **uma vez** e salva o dataset
processado no Drive. Não precisa de GPU.

Tarefas cobertas: **T1** (catálogos/rótulos) · **T2** (download das curvas) · **T3** (preprocessamento + features).

> Status e detalhes de cada tarefa: `docs/PLANO_IMPLEMENTACAO.md`.

In [ ]:
# ============================================================
#  SETUP — funciona no Google Colab e localmente
# ============================================================
import os, sys, subprocess
from pathlib import Path

# 1) Detecta o ambiente
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")

# 2) Instala as libs que não vêm por padrão no Colab
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lightkurve", "astroquery"], check=False)

# 3) Diretório do projeto (no Colab fica no Drive para persistir o cache)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/stellar-classification")
else:
    cwd = Path.cwd()
    PROJECT_DIR = cwd.parent if cwd.name == "notebooks" else cwd

# 4) Caminhos de dados (criados se não existirem)
DATA_DIR      = PROJECT_DIR / "data"
CATALOG_DIR   = DATA_DIR / "catalogs"
RAW_LC_DIR    = DATA_DIR / "raw_lightcurves"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR   = PROJECT_DIR / "results"
for d in (CATALOG_DIR, RAW_LC_DIR, PROCESSED_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 5) Config global do projeto
CLASSES = ["binaria_eclipsante", "pulsante", "modulador_rotacional", "transito_exoplaneta"]
N_BINS = 512          # tamanho do vetor após phase-folding + binning (D4)
N_PER_CLASS = 1000    # alvo de amostras por classe (D3)
RANDOM_STATE = 42

print("PROJECT_DIR:", PROJECT_DIR)
print("Pastas de dados prontas em:", DATA_DIR)

## T1 — Catálogos & montagem de rótulos

Os rótulos físicos vêm de **catálogos dedicados** (o TESS-SVC só classifica por *forma de ajuste*, não por classe física):

| Classe | Fonte | Identificação |
|--------|-------|---------------|
| 🪐 Trânsito de Exoplaneta | **TT9** (VizieR `J/MNRAS/513/102`) | só `Disp == PC` (vetados por humano) |
| 🌗 Binária Eclipsante | **TESS-EBs** (MAST HLSP) | `tess_id` + período + morfologia |
| ✨ Pulsante | **Gaia DR3** `vari_classifier_result` | classes `RR, CEP, DSCT\|GDOR\|SXPHE, BCEP, SPB` |
| 🔄 Modulador Rotacional | **Gaia DR3** `vari_classifier_result` | classes `SOLAR_LIKE, RS` |

Os alvos do Gaia ganham `TIC` via **cross-match posicional** (CDS XMatch → `IV/39/tic82`).
Conflitos entre classes: eclipse-vs-trânsito é ambíguo → descartado; rótulo vetado (TT9/EBs) tem prioridade sobre o estatístico (Gaia).

**Saída:** `data/processed/targets.csv` com `[tic_id, target, ra, dec, label, period_cat, source_catalog]`.
Os catálogos brutos ficam em cache em `data/catalogs/` (re-rodar não re-consulta).

In [ ]:
# ============================================================
#  T1 — funções de aquisição (usa CATALOG_DIR/PROCESSED_DIR/... do SETUP)
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import astropy.units as u
import matplotlib.pyplot as plt

# --- parâmetros do T1 ---
GAIA_SCORE_MIN = 0.85        # confiança mínima da classificação Gaia
GAIA_GMAG_MAX  = 13.0        # estrelas brilhantes -> maior chance de ter curva TESS de 2 min
GAIA_TOP       = 3000        # teto por grupo na consulta Gaia
XMATCH_RADIUS_ARCSEC = 2.0   # raio do cross-match posicional Gaia->TIC
PULSATOR_CLASSES = ["RR", "CEP", "DSCT|GDOR|SXPHE", "BCEP", "SPB"]
ROTATOR_CLASSES  = ["SOLAR_LIKE", "RS"]
PRIORITY = {"binaria_eclipsante": 3, "transito_exoplaneta": 3,
            "pulsante": 1, "modulador_rotacional": 1}  # vetado > estatístico

def _load_or_query(path, query_fn, desc):
    """Cache simples: lê o CSV se existir; senão consulta e salva."""
    if path.exists():
        df = pd.read_csv(path); print(f"  [cache] {desc}: {len(df)} linhas"); return df
    print(f"  [consultando] {desc} ...")
    df = query_fn(); df.to_csv(path, index=False)
    print(f"  [salvo] {desc}: {len(df)} linhas"); return df

def _q_tt9():
    from astroquery.vizier import Vizier
    v = Vizier(row_limit=-1, columns=["**"])
    return v.get_catalogs("J/MNRAS/513/102")["J/MNRAS/513/102/table2"].to_pandas()

def _q_ebs():
    url = ("https://archive.stsci.edu/hlsps/tess-ebs/"
           "hlsp_tess-ebs_tess_lcf-ffi_s0001-s0026_tess_v1.0_cat.csv")
    return pd.read_csv(url)

def _q_gaia(classes):
    from astroquery.gaia import Gaia
    cls = ", ".join(f"'{c}'" for c in classes)
    adql = f"""
        SELECT TOP {GAIA_TOP} c.source_id, c.best_class_name, c.best_class_score,
               s.ra, s.dec, s.phot_g_mean_mag
        FROM gaiadr3.vari_classifier_result AS c
        JOIN gaiadr3.gaia_source AS s ON c.source_id = s.source_id
        WHERE c.best_class_name IN ({cls})
          AND c.best_class_score > {GAIA_SCORE_MIN}
          AND s.phot_g_mean_mag < {GAIA_GMAG_MAX}"""
    return Gaia.launch_job_async(adql).get_results().to_pandas()

def _xmatch_to_tic(df):
    """Cross-match posicional Gaia->TIC (nearest dentro do raio)."""
    from astroquery.xmatch import XMatch
    from astropy.table import Table
    tbl = Table.from_pandas(df[["source_id", "ra", "dec"]].reset_index(drop=True))
    res = XMatch.query(cat1=tbl, cat2="vizier:IV/39/tic82",
                       max_distance=3 * u.arcsec, colRA1="ra", colDec1="dec").to_pandas()
    res = res.sort_values("angDist").drop_duplicates("source_id", keep="first")
    res = res[res["angDist"] < XMATCH_RADIUS_ARCSEC]
    return df.merge(res[["source_id", "TIC"]], on="source_id", how="inner")

def _std(tic, ra, dec, label, period, source):
    return pd.DataFrame({"tic_id": np.asarray(tic), "ra": np.asarray(ra), "dec": np.asarray(dec),
                         "label": label, "period_cat": np.asarray(period), "source_catalog": source})

def _resolve(group):
    """Resolve quando o mesmo TIC aparece com >1 rótulo."""
    labels = set(group["label"])
    if len(labels) == 1:                                          # sem conflito / dup intra-classe
        return group.iloc[[0]]
    if {"binaria_eclipsante", "transito_exoplaneta"} <= labels:   # eclipse vs trânsito: ambíguo
        return group.iloc[0:0]
    g = group.assign(_p=group["label"].map(PRIORITY))
    top = g[g["_p"] == g["_p"].max()]
    if top["label"].nunique() > 1:                                # empate (ex: pulsante vs rotacional)
        return group.iloc[0:0]
    return top.iloc[[0]].drop(columns="_p")

print("Funções do T1 definidas.")

In [ ]:
# ============================================================
#  T1 — executa o pipeline -> data/processed/targets.csv
# ============================================================
print("== T1: aquisição dos catálogos ==")

# (1) Exoplanetas — TT9, somente candidatos vetados (PC)
tt9 = _load_or_query(CATALOG_DIR / "tt9.csv", _q_tt9, "TT9 (VizieR)")
exo = tt9[tt9["Disp"] == "PC"]
exo_df = _std(exo["TIC"], exo["_RA"], exo["_DE"], "transito_exoplaneta", exo["Per"], "TT9")

# (2) Binárias eclipsantes — TESS-EBs
ebs = _load_or_query(CATALOG_DIR / "tess_ebs.csv", _q_ebs, "TESS-EBs (MAST)").drop_duplicates("tess_id")
ebs_df = _std(ebs["tess_id"], ebs["ra"], ebs["dec"], "binaria_eclipsante", ebs["period"], "TESS-EBs")

# (3) Pulsantes — Gaia DR3 + XMatch->TIC
gp = _load_or_query(CATALOG_DIR / "gaia_pulsantes.csv", lambda: _q_gaia(PULSATOR_CLASSES), "Gaia pulsantes")
gp = _load_or_query(CATALOG_DIR / "gaia_pulsantes_tic.csv", lambda: _xmatch_to_tic(gp), "Gaia pulsantes->TIC")
puls_df = _std(gp["TIC"], gp["ra"], gp["dec"], "pulsante", np.nan, "GaiaDR3")

# (4) Moduladores rotacionais — Gaia DR3 + XMatch->TIC
gr = _load_or_query(CATALOG_DIR / "gaia_rotacionais.csv", lambda: _q_gaia(ROTATOR_CLASSES), "Gaia rotacionais")
gr = _load_or_query(CATALOG_DIR / "gaia_rotacionais_tic.csv", lambda: _xmatch_to_tic(gr), "Gaia rotacionais->TIC")
rota_df = _std(gr["TIC"], gr["ra"], gr["dec"], "modulador_rotacional", np.nan, "GaiaDR3")

# --- consolida, resolve conflitos, balanceia ---
allc = pd.concat([exo_df, ebs_df, puls_df, rota_df], ignore_index=True).dropna(subset=["tic_id"]).copy()
allc["tic_id"] = allc["tic_id"].astype("int64")
clean = (allc.groupby("tic_id", group_keys=False)[allc.columns.tolist()]
              .apply(_resolve).reset_index(drop=True))

parts = [grp.sample(n=min(N_PER_CLASS, len(grp)), random_state=RANDOM_STATE)
         for _, grp in clean.groupby("label")]
targets = pd.concat(parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
targets["target"] = "TIC " + targets["tic_id"].astype(str)
targets = targets[["tic_id", "target", "ra", "dec", "label", "period_cat", "source_catalog"]]
targets.to_csv(PROCESSED_DIR / "targets.csv", index=False)

# --- resumo + gráfico ---
print("\n== T1: resultado ==")
print(targets["label"].value_counts().to_string())
print("\nTotal de alvos:", len(targets), "-> salvo em", PROCESSED_DIR / "targets.csv")

counts = targets["label"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(counts.index, counts.values, color=["#4C72B0", "#55A868", "#C44E52", "#8172B3"])
for i, v in enumerate(counts.values): ax.text(v + 5, i, str(v), va="center")
ax.set_xlabel("nº de alvos"); ax.set_title("T1 — alvos por classe")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "t1_class_counts.png", dpi=120); plt.show()
targets.head()

## T2 — Download das curvas de luz (cache resumível)

Para cada `TIC` de `targets.csv`, busca a curva de luz TESS via `lightkurve` e salva
`(time, flux, flux_err)` em `data/raw_lightcurves/<tic>.npz`.

- **Preferência de produto:** SPOC 2 min (PDCSAP) → SPOC 20 s → TESS-SPOC (FFI) → QLP. Flux preferido = PDCSAP.
- **Download isolado por alvo** (cada um em seu diretório temporário) → evita corromper o cache com vários downloads em paralelo.
- **Resumível:** pula TICs já baixados (npz existe) ou registrados no `download_manifest.csv`. Pode interromper e retomar.
- **Paralelo:** `ThreadPoolExecutor` (`N_WORKERS`); `socket` timeout de 60 s evita travas longas.

> Medido (protótipo, 100% sucesso): ~5.9 s/curva, ~110 KB/curva → **~400 MB** no total.
> **Cobertura esperada ~90–94%** (alguns alvos fracos não têm curva TESS). ⚠️ Se a barra **parar de avançar** (um punhado de alvos "problemáticos" pendura a conexão): **interrompa o kernel e rode a célula de novo** — é resumível e continua de onde parou. Não precisa de 100%.
> ⚠️ Etapa longa e que precisa do **Google Drive** (no Colab). Rode em runtime **CPU**.

In [ ]:
# ============================================================
#  T2 — funções de download (lightkurve)
# ============================================================
import tempfile, shutil, socket
import lightkurve as lk
lk.log.setLevel("ERROR")
socket.setdefaulttimeout(60)                 # nenhuma operação de rede trava > 60s

FITS_TMP_ROOT = RAW_LC_DIR / "_fits_tmp"     # cada download usa um subdir isolado (evita corromper cache em paralelo)
FITS_TMP_ROOT.mkdir(exist_ok=True)

# ordem de preferência (autor, exptime). None = qualquer exptime daquele autor.
PREFS = [("SPOC", 120), ("SPOC", 20), ("TESS-SPOC", None), ("QLP", None)]
MIN_POINTS = 50                              # descarta curvas curtas demais
TERMINAL = {"ok", "no_data", "too_short", "dl_none", "cache"}   # status que não re-tenta

def _pick_index(tbl):
    auth = np.array(tbl["author"], dtype=str)
    exp  = np.array(tbl["exptime"], dtype=float)
    seq  = np.array(tbl["sequence_number"], dtype=float)
    for a, e in PREFS:
        m = (auth == a)
        if e is not None: m &= np.isclose(exp, e)
        idx = np.where(m)[0]
        if len(idx):
            best = idx[np.argmin(seq[idx])]          # setor mais antigo
            return int(best), auth[best], float(exp[best])
    return 0, auth[0], float(exp[0])                 # fallback: 1º produto

def _get_flux(lc):
    if "pdcsap_flux" in lc.colnames:                 # SPOC/TESS-SPOC: flux corrigido (PDCSAP)
        f = np.asarray(lc["pdcsap_flux"].value, dtype=float)
        fe = (np.asarray(lc["pdcsap_flux_err"].value, dtype=float)
              if "pdcsap_flux_err" in lc.colnames else np.full_like(f, np.nan))
        return f, fe, "pdcsap_flux"
    for c in ("sap_flux", "kspsap_flux", "det_flux", "cal_flux"):
        if c in lc.colnames:
            f = np.asarray(getattr(lc[c], "value", lc[c]), dtype=float)
            ec = c + "_err"
            fe = (np.asarray(getattr(lc[ec], "value", lc[ec]), dtype=float)
                  if ec in lc.colnames else np.full_like(f, np.nan))
            return f, fe, c
    return (np.asarray(lc.flux.value, dtype=float),
            np.asarray(lc.flux_err.value, dtype=float), "flux")

def _fetch_one(tic, retries=2):
    """Baixa 1 setor em diretório isolado; salva npz. Retorna dict de status (resumível)."""
    npz = RAW_LC_DIR / f"{tic}.npz"
    if npz.exists():
        return {"tic_id": tic, "status": "cache"}
    last = "error"
    for _ in range(retries + 1):
        tmp = tempfile.mkdtemp(dir=str(FITS_TMP_ROOT))     # dir isolado por tentativa
        try:
            sr = lk.search_lightcurve(f"TIC {tic}", mission="TESS")
            if len(sr) == 0:
                return {"tic_id": tic, "status": "no_data"}
            i, author, exptime = _pick_index(sr.table)
            lc = sr[i].download(download_dir=tmp)
            if lc is None:
                last = "dl_none"; continue
            t = np.asarray(lc.time.value, dtype=float)
            f, fe, fcol = _get_flux(lc)
            m = np.isfinite(t) & np.isfinite(f)
            t, f, fe = t[m], f[m], fe[m]
            if len(t) < MIN_POINTS:
                return {"tic_id": tic, "status": "too_short", "n_points": int(len(t))}
            sector = int(sr.table["sequence_number"][i])
            np.savez_compressed(npz, time=t, flux=f, flux_err=fe, author=str(author),
                                sector=sector, exptime=float(exptime), fluxcol=fcol)
            return {"tic_id": tic, "status": "ok", "author": str(author), "sector": sector,
                    "exptime": float(exptime), "fluxcol": fcol, "n_points": int(len(t))}
        except Exception as e:
            last = f"{type(e).__name__}: {str(e)[:80]}"
        finally:
            shutil.rmtree(tmp, ignore_errors=True)         # limpa o FITS na hora
    return {"tic_id": tic, "status": "error", "error": last}

print("Funções do T2 definidas. (download isolado por alvo, timeout de rede 60s)")

In [ ]:
# ============================================================
#  T2 — executa o download paralelo e resumível
# ============================================================
import os, contextlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

N_WORKERS = 5                                  # com download isolado é estável; pode subir p/ 8 se a rede aguentar
MANIFEST  = PROCESSED_DIR / "download_manifest.csv"

targets = pd.read_csv(PROCESSED_DIR / "targets.csv")
label_map = dict(zip(targets["tic_id"], targets["label"]))

# o que já está pronto (resumível): npz no disco + manifest com status terminal
man_rows, done = [], set()
if MANIFEST.exists():
    prev = pd.read_csv(MANIFEST)
    man_rows = prev.to_dict("records")
    done |= set(prev.loc[prev["status"].isin(TERMINAL), "tic_id"].astype(int))
done |= {int(p.stem) for p in RAW_LC_DIR.glob("*.npz")}

todo = [int(t) for t in targets["tic_id"] if int(t) not in done]
print(f"Total {len(targets)} | já prontos {len(targets)-len(todo)} | a baixar {len(todo)}")

# stdout redirecionado p/ silenciar logs do lightkurve/astroquery (a barra do tqdm usa stderr)
with open(os.devnull, "w") as _dn, contextlib.redirect_stdout(_dn):
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = {ex.submit(_fetch_one, t): t for t in todo}
        for k, fut in enumerate(tqdm(as_completed(futs), total=len(futs),
                                     desc="download", file=sys.stderr)):
            r = fut.result(); r["label"] = label_map.get(r["tic_id"])
            man_rows.append(r)
            if (k + 1) % 50 == 0:              # checkpoint do manifest (resumível)
                pd.DataFrame(man_rows).drop_duplicates("tic_id", keep="last").to_csv(MANIFEST, index=False)

manifest = pd.DataFrame(man_rows).drop_duplicates("tic_id", keep="last")
manifest.to_csv(MANIFEST, index=False)

print("\nStatus final:", manifest["status"].value_counts().to_dict())
print(f"Curvas em cache: {len(list(RAW_LC_DIR.glob('*.npz')))}")
print("Cobertura por classe (%):")
cov = (manifest.assign(ok=manifest["status"].isin(["ok", "cache"]))
                .groupby("label")["ok"].mean().mul(100).round(1))
print(cov.to_string())

In [ ]:
# (opcional) limpa os FITS temporários — o .npz já tem time/flux/flux_err
import shutil
if FITS_TMP_ROOT.exists():
    shutil.rmtree(FITS_TMP_ROOT); FITS_TMP_ROOT.mkdir(exist_ok=True)
    print("Diretório temporário de FITS limpo.")
npz_files = list(RAW_LC_DIR.glob("*.npz"))
print(f"{len(npz_files)} curvas em cache, {sum(p.stat().st_size for p in npz_files)/1e6:.0f} MB em data/raw_lightcurves/")

## T3 — Preprocessamento & extração de features

Transforma cada curva crua num vetor de tamanho fixo (512) + um conjunto de *features* clássicas.

**Decisão importante:** o *flattening* agressivo do documento serve para *detectar trânsitos*, mas **apagaria** o sinal de pulsação/rotação que queremos classificar. Então o pipeline **preserva a variabilidade**:

1. **Normaliza** (fluxo ÷ mediana → ~1).
2. **Sigma-clip assimétrico** (só outliers superiores, via MAD): remove raios cósmicos, **preserva quedas** (trânsitos/eclipses).
3. **Período**: usa o do catálogo (TT9/TESS-EBs) quando existe; senão **Lomb-Scargle**.
4. **Phase-folding** no período + **binning por mediana em 512 pontos** (isso "auto-detrenda" o sinal dobrado e reduz ruído).
5. **Centraliza** a feature principal (mínimo suavizado) e aplica **`arcsinh`** em torno da base 1.0 → entrada da CNN. O `arcsinh` **preserva a profundidade/amplitude** (trânsito raso e eclipse profundo ficam distinguíveis) ao mesmo tempo que comprime a faixa dinâmica. *(Padronização por z-score apagava a amplitude e prejudicava muito a CNN.)*

Em paralelo extrai *features* para os modelos clássicos: período, potência de Lomb-Scargle, amplitude, profundidade, skewness, kurtosis, etc.

**Saídas:** `X.npy` (N×512, entrada da CNN), `y.npy` (rótulo inteiro), `ids.npy` (TIC), `features.csv` (entrada do RF). Gráfico de sanidade em `results/`.

In [ ]:
# ============================================================
#  T3 — funções de preprocessamento
# ============================================================
from astropy.stats import sigma_clip, mad_std
from astropy.timeseries import LombScargle
from scipy.stats import skew, kurtosis, binned_statistic
from scipy.ndimage import median_filter

PMIN, PMAX = 0.05, 20.0          # faixa de períodos procurados pelo Lomb-Scargle (dias)

def ls_period_power(t, f):
    """Período e potência dominantes via Lomb-Scargle."""
    freq, power = LombScargle(t, f).autopower(minimum_frequency=1/PMAX,
                                              maximum_frequency=1/PMIN, samples_per_peak=5)
    i = int(np.argmax(power))
    return 1.0 / freq[i], float(power[i])

def preprocess_one(t, f, period_cat=None):
    """Curva crua -> (forma[512] p/ CNN, dict de features). None se inutilizável."""
    f = f / np.nanmedian(f)                                          # normaliza ~1
    keep = ~sigma_clip(f, sigma_lower=np.inf, sigma_upper=5,         # remove só spikes (raios cósmicos)
                       cenfunc="median", stdfunc="mad_std").mask
    t, f = t[keep], f[keep]
    if len(f) < 50:
        return None, None
    ls_p, ls_pow = ls_period_power(t, f)
    use_cat = period_cat is not None and np.isfinite(period_cat) and period_cat > 0
    P = float(period_cat) if use_cat else ls_p
    phase = ((t - t.min()) / P) % 1.0
    binned, _, _ = binned_statistic(phase, f, statistic="median", bins=N_BINS, range=(0, 1))
    good = np.isfinite(binned)
    if good.sum() < 10:
        return None, None
    binned = np.interp(np.arange(N_BINS), np.flatnonzero(good), binned[good])   # preenche bins vazios
    binned = median_filter(binned, size=3, mode="nearest")                      # tira spikes de 1 bin
    cmin = int(np.argmin(median_filter(binned, size=11, mode="nearest")))       # mínimo robusto
    binned = np.roll(binned, N_BINS // 2 - cmin)                                # centraliza feature principal
    # representação que PRESERVA profundidade/amplitude (chave p/ a CNN): arcsinh em torno da
    # base 1.0 na escala de 1% -> trânsito raso (~-0.5) e eclipse profundo (~-4) ficam visíveis.
    shape = np.arcsinh((binned - 1.0) / 0.01).astype(np.float32)
    depth = float(1.0 - binned.min())
    feats = dict(period=P, used_ls=int(not use_cat), ls_power=ls_pow,
                 amp_p95_p05=float(np.percentile(f, 95) - np.percentile(f, 5)),
                 amp_minmax=float(f.max() - f.min()), std=float(np.std(f)),
                 mad=float(mad_std(f)), skew=float(skew(f)), kurt=float(kurtosis(f)),
                 depth=depth, frac_below_half=float(np.mean(binned < (1 - depth / 2))),
                 n_points=int(len(f)))
    return shape, feats

print("Funções do T3 definidas. N_BINS =", N_BINS)

In [ ]:
# ============================================================
#  T3 — processa todas as curvas -> X.npy, y.npy, ids.npy, features.csv
# ============================================================
from tqdm.auto import tqdm

targets = pd.read_csv(PROCESSED_DIR / "targets.csv").set_index("tic_id")
cached = sorted(RAW_LC_DIR.glob("*.npz"))
print(f"Curvas em cache: {len(cached)}")

X, ids, feat_rows, examples = [], [], [], {}
for p in tqdm(cached, desc="preprocess"):
    tic = int(p.stem)
    if tic not in targets.index:
        continue
    z = np.load(p, allow_pickle=True)
    row = targets.loc[tic]
    shape, feats = preprocess_one(z["time"], z["flux"], row["period_cat"])
    if shape is None:
        continue
    X.append(shape); ids.append(tic)
    feats["tic_id"] = tic; feats["label"] = row["label"]; feat_rows.append(feats)
    examples.setdefault(row["label"], []).append((tic, shape))

X = np.asarray(X, dtype=np.float32)
features = pd.DataFrame(feat_rows)
y = np.array([CLASSES.index(l) for l in features["label"]], dtype=np.int64)
ids = np.asarray(ids, dtype=np.int64)

np.save(PROCESSED_DIR / "X.npy", X)
np.save(PROCESSED_DIR / "y.npy", y)
np.save(PROCESSED_DIR / "ids.npy", ids)
features.to_csv(PROCESSED_DIR / "features.csv", index=False)

print(f"\nX: {X.shape} | NaN: {int(np.isnan(X).sum())} | classes (y): {np.bincount(y)}")
print("Salvos em data/processed/: X.npy, y.npy, ids.npy, features.csv")
print("\nFeatures (média por classe):")
print(features.groupby("label")[["period", "depth", "amp_p95_p05", "ls_power", "skew"]].mean().round(3).to_string())

# gráfico de sanidade: 3 exemplos dobrados por classe
order = ["transito_exoplaneta", "binaria_eclipsante", "pulsante", "modulador_rotacional"]
fig, axes = plt.subplots(len(order), 3, figsize=(11, 9))
for r, lab in enumerate(order):
    ex = examples.get(lab, [])[:3]
    for c in range(3):
        ax = axes[r, c]
        if c < len(ex):
            ax.plot(ex[c][1], lw=0.8); ax.set_title(f"{lab}\nTIC {ex[c][0]}", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("T3 — curvas dobradas e padronizadas (sanidade por classe)", fontsize=11)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "t3_sanity_folded.png", dpi=120); plt.show()